In [1]:
%pip install ibm-watsonx-ai ibm-cos-sdk

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.5/4.5 MB 46.3 MB/s  0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Installing backend dependencies ... done
  Preparing metadata (pyproject.toml) ... done
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 50.1 MB/s  0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 51.4 MB/s  0:00:00
  Created wheel for ibm-cos-sdk: filename=ibm_cos_sdk-2.

In [1]:
import os, json, importlib.util
from pathlib import Path
import numpy as np, pandas as pd, joblib

ROOT = Path.cwd().parent if Path.cwd().name == "notebook" else Path.cwd()
RAW, MOD, OUT = ROOT / "data" / "raw", ROOT / "models", ROOT / "outputs"
OUT.mkdir(parents=True, exist_ok=True)

def load_env(path):
    if not path.exists():
        return False
    for line in path.read_text().splitlines():
        line = line.strip()
        if line and not line.startswith("#") and "=" in line:
            k, v = line.split("=", 1); os.environ[k.strip()] = v.strip().strip('"').strip("'")
    return True

print(".env found:", load_env(ROOT / ".env"))
for pkg in ["ibm_watsonx_ai", "ibm_boto3", "transformers", "torch"]:
    print(f"{pkg:15}", "installed" if importlib.util.find_spec(pkg) else "NOT installed")
print("models present:", [p.name for p in MOD.glob("*.joblib")])

.env found: False
ibm_watsonx_ai  installed
ibm_boto3       installed
transformers    installed
torch           installed
models present: ['features.joblib', 'ttf_reg.joblib', 'failure_clf.joblib']


In [5]:
# Cell 2 (replacement): auto-finds a scenario and hour where zones are still healthy but at high risk
FEATURES = joblib.load(MOD / "features.joblib")
clf, reg = joblib.load(MOD / "failure_clf.joblib"), joblib.load(MOD / "ttf_reg.joblib")
pq, csv = RAW / "sim_runs.parquet", RAW / "sim_runs.csv"
df = pd.read_parquet(pq) if pq.exists() else pd.read_csv(csv)

THRESH = 0.3          # risk cutoff for "high risk"

def add_features(d):
    d = d.sort_values(["node", "hour"]).reset_index(drop=True) 
    g = d.groupby("node")
    d["depth_rise_1h"] = d["flood_depth"] - g["flood_depth"].shift(1)
    d["depth_rise_3h"] = d["flood_depth"] - g["flood_depth"].shift(3)
    d["requests_trend"] = d["requests"] - g["requests"].shift(3)
    d["supply_burn_3h"] = g["days_of_supply"].shift(3) - d["days_of_supply"]
    d["occ_trend"] = d["nearest_shelter_occ"] - g["nearest_shelter_occ"].shift(3)
    d["road_loss_3h"] = g["open_road_frac"].shift(3) - d["open_road_frac"]
    rain = d.drop_duplicates("hour").set_index("hour")["rain_mm"].sort_index()
    d["rain_3h"] = d["hour"].map(rain.rolling(3, min_periods=1).sum())
    d["rain_6h"] = d["hour"].map(rain.rolling(6, min_periods=1).sum())
    d[FEATURES] = d[FEATURES].fillna(0).astype("float32")
    d["risk_6h"] = clf.predict_proba(d[FEATURES])[:, 1]
    return d

# candidate scenarios: spread from moderate to severe (or the one you choose)
fr = df.groupby("scenario")["failed_now"].mean().sort_values()
SCENARIO = None       # set a number (e.g. 7) to force a scenario, or leave None to search
HOUR = None           # set a number to force an hour, or leave None to search
cands = [SCENARIO] if SCENARIO is not None else list(fr.index[int(len(fr) * 0.4):][::4])   # ~30 scenarios

best, scored = None, {}
for s in cands:
    d = add_features(df[df["scenario"] == s])
    scored[s] = d
    hours = [HOUR] if HOUR is not None else range(6, 60)
    for h in hours:
        sub = d[(d["hour"] == h) & (d["failed_now"] == 0)]
        n_high = int((sub["risk_6h"] >= THRESH).sum())
        # prefer many high-risk zones AND a mix of failed/healthy zones for a realistic story
        n_failed = int(((d["hour"] == h) & (d["failed_now"] == 1)).sum())
        score = -abs(n_high - 15) - 0.2 * max(0, n_failed - 10)
        if best is None or score > best[0]:
            best = (score, s, h, n_high)

if best is None or best[3] == 0:
    THRESH = 0.1
    print("No zone reached 0.3 anywhere. Lowering the cutoff to 0.1.")
    best = None
    for s, d in scored.items():
        for h in ([HOUR] if HOUR is not None else range(6, 60)):
            n_high = int(((d["hour"] == h) & (d["failed_now"] == 0) & (d["risk_6h"] >= THRESH)).sum())
            if best is None or n_high > best[3]:
                best = (n_high, s, h, n_high)

_, SCENARIO, HOUR, _ = best
d = scored[SCENARIO]
print(f"Chosen: scenario {SCENARIO}, hour {HOUR}, cutoff {THRESH}")

now = d[d["hour"] == HOUR].copy()
now["est_hours_to_failure"] = np.clip(reg.predict(now[FEATURES]), 0, 72)

def main_driver(r):
    if r["isolated"] == 1: return "cut off from all shelters"
    if r["days_of_supply"] < 1: return "supplies running low"
    if r["nearest_shelter_occ"] > 0.8: return "nearest shelter near capacity"
    return "rising flood water"
now["main_driver"] = now.apply(main_driver, axis=1)

at_risk = now[now["failed_now"] == 0].sort_values(["risk_6h", "population"], ascending=False)
summary = {"scenario": int(SCENARIO), "hour": int(HOUR), "zones_total": int(len(now)),
           "zones_already_failed": int((now["failed_now"] == 1).sum()),
           "zones_isolated": int(now["isolated"].sum()),
           "zones_high_risk_next_6h": int((at_risk["risk_6h"] >= THRESH).sum()),
           "people_in_high_risk_zones": int(at_risk.loc[at_risk["risk_6h"] >= THRESH, "population"].sum())}
cols = ["node", "population", "risk_6h", "est_hours_to_failure", "flood_depth",
        "days_of_supply", "nearest_shelter_occ", "isolated", "main_driver"]
top = at_risk[cols].head(8).round(2).reset_index(drop=True)
print(summary)
top

Chosen: scenario 149, hour 31, cutoff 0.3
{'scenario': 149, 'hour': 31, 'zones_total': 53, 'zones_already_failed': 1, 'zones_isolated': 0, 'zones_high_risk_next_6h': 15, 'people_in_high_risk_zones': 32592}


,node,population,risk_6h,est_hours_to_failure,flood_depth,days_of_supply,nearest_shelter_occ,isolated,main_driver
0,3_4,1433.0,1.00,2.01,0.24,0.31,0.00,0,supplies running low
1,3_5,1368.0,1.00,4.87,0.23,0.43,0.00,0,supplies running low
2,7_5,1490.0,0.84,2.88,0.40,2.84,0.12,0,rising flood water
3,7_6,3353.0,0.69,3.91,0.43,3.53,0.12,0,rising flood water
4,6_5,3485.0,0.66,4.05,0.36,3.37,0.12,0,rising flood water
5,5_2,3491.0,0.46,14.15,0.22,1.59,0.10,0,rising flood water
6,7_1,3552.0,0.45,6.44,0.39,1.25,0.05,0,rising flood water
7,7_3,919.0,0.44,4.13,0.35,4.56,0.10,0,rising flood water


In [7]:
prompt = f"""You are a disaster-response advisor writing for emergency coordinators.
Using ONLY the data below, write: 1) a 4-line situation report, 2) the top 3 recommended actions,
most urgent first, naming specific zones. Do not invent numbers.
SUMMARY: {json.dumps(summary)}
HIGHEST-RISK ZONES (risk_6h = probability of failure within 6 hours):
{top.to_json(orient="records")}"""

def template_report():
    lines = [f"SITUATION (scenario {SCENARIO}, hour {HOUR}): {summary['zones_already_failed']} of "
             f"{summary['zones_total']} zones have failed, {summary['zones_isolated']} are cut off. "
             f"{summary['zones_high_risk_next_6h']} more zones ({summary['people_in_high_risk_zones']:,} people) "
             f"are at high risk within 6 hours.", "", "TOP ACTIONS:"]
    for i, r in top.head(3).iterrows():
        lines.append(f"{i+1}. Zone {r['node']} (pop {int(r['population']):,}, risk {r['risk_6h']:.0%}, "
                     f"about {r['est_hours_to_failure']:.0f} h to failure): {r['main_driver']}.")
    return "\n".join(lines)

def try_watsonx():
    if not (os.getenv("WATSONX_APIKEY") and os.getenv("WATSONX_PROJECT_ID")):
        raise RuntimeError("watsonx credentials missing")
    from ibm_watsonx_ai import Credentials
    from ibm_watsonx_ai.foundation_models import ModelInference
    creds = Credentials(url=os.getenv("WATSONX_URL", "https://us-south.ml.cloud.ibm.com"), api_key=os.environ["WATSONX_APIKEY"])
    last = None
    for mid in ["ibm/granite-3-3-8b-instruct", "ibm/granite-3-8b-instruct", "ibm/granite-4-h-small"]:
        try:
            m = ModelInference(model_id=mid, credentials=creds, project_id=os.environ["WATSONX_PROJECT_ID"])
            return m.generate_text(prompt=prompt, params={"max_new_tokens": 350, "temperature": 0}), f"watsonx.ai ({mid})"
        except Exception as e:
            last = e
    raise RuntimeError(f"no watsonx model worked: {type(last).__name__}")

LOCAL_GRANITE = ROOT / "granite-4.0-h-350m"     # only used if this folder exists
def try_local():
    if not (LOCAL_GRANITE / "config.json").exists():
        raise RuntimeError("no local model folder")
    from transformers import AutoTokenizer, AutoModelForCausalLM
    tok, mdl = AutoTokenizer.from_pretrained(LOCAL_GRANITE), AutoModelForCausalLM.from_pretrained(LOCAL_GRANITE)
    ids = tok(prompt, return_tensors="pt")
    out = mdl.generate(**ids, max_new_tokens=250, do_sample=False)
    return tok.decode(out[0][ids["input_ids"].shape[1]:], skip_special_tokens=True), "local Granite 350M"

report, backend = None, None
for fn in (try_watsonx, try_local):
    try:
        report, backend = fn(); break
    except Exception as e:
        print(f"{fn.__name__} skipped: {type(e).__name__}: {str(e)[:100]}")
if not report or len(report.strip()) < 40:
    report, backend = template_report(), "template (no LLM available)"
print("REPORT SOURCE:", backend, "\n"); print(report)

try_watsonx skipped: RuntimeError: watsonx credentials missing
try_local skipped: RuntimeError: no local model folder
REPORT SOURCE: template (no LLM available) 

SITUATION (scenario 149, hour 31): 1 of 53 zones have failed, 0 are cut off. 15 more zones (32,592 people) are at high risk within 6 hours.

TOP ACTIONS:
1. Zone 3_4 (pop 1,433, risk 100%, about 2 h to failure): supplies running low.
2. Zone 3_5 (pop 1,368, risk 100%, about 5 h to failure): supplies running low.
3. Zone 7_5 (pop 1,490, risk 84%, about 3 h to failure): rising flood water.


In [8]:
(OUT / "situation_report.md").write_text(f"# FloodShield situation report\n\n_Source: {backend}_\n\n{report}\n")
at_risk[cols].round(3).to_csv(OUT / "risk_forecast.csv", index=False)
(OUT / "forecast_summary.json").write_text(json.dumps(summary, indent=2))
files = [f for f in [OUT / "situation_report.md", OUT / "risk_forecast.csv", OUT / "forecast_summary.json"]
         + list(MOD.glob("*.joblib")) + list((OUT / "figures").glob("*.png")) if f.exists()]
print("saved locally:", len(files), "files in", OUT)

need = ["COS_APIKEY", "COS_INSTANCE_ID", "COS_ENDPOINT", "COS_BUCKET"]
if all(os.getenv(k) for k in need) and importlib.util.find_spec("ibm_boto3"):
    try:
        import ibm_boto3
        from ibm_botocore.client import Config
        cos = ibm_boto3.client("s3", ibm_api_key_id=os.environ["COS_APIKEY"],
                               ibm_service_instance_id=os.environ["COS_INSTANCE_ID"],
                               config=Config(signature_version="oauth"), endpoint_url=os.environ["COS_ENDPOINT"])
        for f in files:
            cos.upload_file(str(f), os.environ["COS_BUCKET"], f"floodshield/{f.name}")
        print("uploaded", len(files), "objects to bucket", os.environ["COS_BUCKET"])
    except Exception as e:
        print("COS upload failed:", type(e).__name__, str(e)[:200], "| files remain in outputs/")
else:
    print("COS not configured (credentials or package missing). Files stay in", OUT)

saved locally: 14 files in /home/ibm-user/floodguard/outputs
COS not configured (credentials or package missing). Files stay in /home/ibm-user/floodguard/outputs
